# Function 8 Analysis - Week 7

**Function description:** You're optimising an eight-dimensional black-box function; each parameter influences the output (e.g., performance/accuracy), but the mechanism is unknown. The goal is to maximise the response, so we rely on BO to navigate a likely multi-modal surface while prioritising strong local maxima.

**New datapoint (Week 7):** `(0.050000, 0.050000, 0.050000, 0.050000, 0.600000, 0.930000, 0.306800, 0.700000)` returned **≈9.72709**, slightly below last week’s 9.7497 peak—**not a new max**. Total observations: **46**.

**Analysis and why we chose it last week:** Following last week’s (Week 6) plan, we widened x7/x8 to test whether the ridge extends past the x7≈0.21, x8≈0.70 sweet spot while keeping x1–x6 fixed at the best template. Pushing x7 up to 0.3068 with x8 held at 0.70 dipped to 9.727, suggesting the peak remains near x7≈0.21 with x8≈0.70 and that higher x7 provides no uplift. x1–x6 stability still appears valid; x5 mid/high remains fine.

**Recommendation for this week’s BO changes:** Keep EI with `xi` ≈0.02, continue fixing x1–x6 but permit a small x5 perturbation (±0.02) in restarts, narrow x7 to **[0.18, 0.28]** (center ~0.21) and x8 to **[0.68, 0.74]**, and add a light jitter/repulsion around `(x7, x8) ≈ (0.21, 0.70)` to avoid re-sampling the identical point. Prioritise EI restarts in that band; only a few global restarts are needed.


## Loading and Displaying the Data

We load the inputs and outputs for function 8. Week 7 `(0.05, 0.05, 0.05, 0.05, 0.6, 0.93, 0.3068, 0.7000)` returned **≈9.7271 (not a max)**, so last week’s `(0.050000, 0.050000, 0.050000, 0.050000, 0.600000, 0.930000, 0.210300, 0.700000)` remains best at ≈9.7497. Earlier: Week 2 ≈9.73, Week 3 (x7=0) dropped to 9.6549, Week 4 (x7=0.10) rebounded to 9.72465. x7 and x8 remain the key levers.


In [5]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="ticks", context="notebook")
path = Path("../../initial_data/function_8")
X = np.load(path / "initial_inputs.npy")
y = np.load(path / "initial_outputs.npy")

# Add the new points from Week 1–6
X_new_point_week_1 = np.array([[0.050000, 0.050000, 0.050000, 0.050000, 0.600000, 0.930000, 0.250000, 0.450000]])
y_new_point_week_1 = np.array([9.74365])
X_new_point_week_2 = np.array([[0.100000, 0.100000, 0.100000, 0.100000, 0.100000, 0.400000, 0.250000, 0.450000]])
y_new_point_week_2 = np.array([9.73005])
X_new_point_week_3 = np.array([[0.050000, 0.050000, 0.050000, 0.050000, 0.600000, 0.930000, 0.000000, 1.000000]])
y_new_point_week_3 = np.array([9.6549])
X_new_point_week_4 = np.array([[0.050000, 0.050000, 0.050000, 0.050000, 0.600000, 0.930000, 0.100000, 0.350000]])
y_new_point_week_4 = np.array([9.72465])
X_new_point_week_5 = np.array([[0.050000, 0.050000, 0.050000, 0.050000, 0.600000, 0.930000, 0.210300, 0.700000]])
y_new_point_week_5 = np.array([9.74968782])
X_new_point_week_6 = np.array([[0.050000, 0.050000, 0.050000, 0.050000, 0.600000, 0.930000, 0.306800, 0.700000]])
y_new_point_week_6 = np.array([9.72708752])

X = np.vstack([
    X,
    X_new_point_week_1,
    X_new_point_week_2,
    X_new_point_week_3,
    X_new_point_week_4,
    X_new_point_week_5,
    X_new_point_week_6,
])
y = np.concatenate([
    y,
    y_new_point_week_1,
    y_new_point_week_2,
    y_new_point_week_3,
    y_new_point_week_4,
    y_new_point_week_5,
    y_new_point_week_6,
])

df = pd.DataFrame(X, columns=["x1", "x2", "x3", "x4", "x5", "x6", "x7", "x8"]); df["y"] = y
display(df)
print("df sorted by y")
df_sorted = df.sort_values("y", ascending=False).reset_index(drop=True)
display(df_sorted)


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


df sorted by y


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.050000,0.050000,0.050000,0.050000,0.600000,0.930000,0.210300,0.700000,9.749688
1,0.050000,0.050000,0.050000,0.050000,0.600000,0.930000,0.250000,0.450000,9.743650
2,0.100000,0.100000,0.100000,0.100000,0.100000,0.400000,0.250000,0.450000,9.730050
3,0.050000,0.050000,0.050000,0.050000,0.600000,0.930000,0.306800,0.700000,9.727088
4,0.050000,0.050000,0.050000,0.050000,0.600000,0.930000,0.100000,0.350000,9.724650
5,0.050000,0.050000,0.050000,0.050000,0.600000,0.930000,0.000000,1.000000,9.654900
6,0.056447,0.065956,0.022929,0.038786,0.403935,0.801055,0.488307,0.893085,9.598482
7,0.192640,0.630677,0.416796,0.490529,0.796086,0.654567,0.276241,0.295518,9.344274
8,0.481245,0.102461,0.219486,0.677322,0.247509,0.244341,0.163825,0.715962,9.183005
9,0.145120,0.119328,0.420888,0.387609,0.155423,0.875172,0.510560,0.728611,9.141639


## Bayesian Optimization Setup

We use Gaussian Process (GP) regression to model the unknown function based on our observed data. The GP provides both a mean prediction and uncertainty estimates. The search space is defined as [0, 1] for each of the eight input variables.

**Strategy Evolution:**
- **Week 1:** Used UCB to find a point with low x1-x4 and moderate x5-x8, which scored ≈9.74.
- **Week 2:** Continued with UCB, which found a point with very different x1-x6 values but similar x7 (0.25) and x8 (0.45), scoring ≈9.73 (second highest). This suggests x7 and x8 are the key features.
- **Week 3:** Given that x7 and x8 appear to be the critical features, we implement an **exploitation strategy** that:
  - Keeps x1-x6 fixed at the best point's values
  - Focuses optimization on x7 and x8 only
  - Uses Expected Improvement (EI) with low exploration to exploit the promising x7-x8 region
- **Week 7:** `(0.05, ..., x7=0.3068, x8=0.70)` scored ≈9.727 (not a max); the peak remains at x7≈0.21, x8≈0.70. Recommendation for next BO step: keep x1–x6 fixed, run EI with ξ≈0.02 over a **narrow x7 band (0.18–0.26)** and x8 band (0.68–0.74) with a jitter penalty to avoid re-hitting 0.2103 exactly.

In [6]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
np.random.seed(42)
# Per-feature lengthscales with bounds (assuming little noise, no WhiteKernel)
kernel = Matern(
    length_scale=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    length_scale_bounds=(0.2, 5.0),
    nu=2.5
)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5)
gp.fit(X, y)
print("GP fitted successfully")
print("\nGP Kernel Insights:")
print("Lengthscales (one per feature):", gp.kernel_.length_scale)
print("Full kernel parameters:", gp.kernel_.get_params())


GP fitted successfully

GP Kernel Insights:
Lengthscales (one per feature): [1.27978655 1.93172552 0.94242713 2.40476565 4.9429576  5.
 1.38846268 5.        ]
Full kernel parameters: {'length_scale': array([1.27978655, 1.93172552, 0.94242713, 2.40476565, 4.9429576 ,
       5.        , 1.38846268, 5.        ]), 'length_scale_bounds': (0.2, 5.0), 'nu': 2.5}


d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


## Finding the Next Point to Evaluate (Week 7 config)

We keep EI but lift `xi` to **0.02**, keep x1–x4/x6 fixed at the best template, allow a small **x5 jitter (±0.02)**, and narrow x7/x8 to the observed sweet band. Strategy:
- Fix x1–x4, x6 to the best point; allow x5 ∈ [0.58, 0.62] during search
- Optimize x5/x7/x8 jointly with EI (`xi=0.02`), with x7 ∈ [0.18, 0.28], x8 ∈ [0.68, 0.74]
- Bias restarts near (x7, x8) ≈ (0.21, 0.70), add a light repulsion/jitter on (x7, x8) to avoid re-sampling the exact same corner


In [7]:
from scipy.stats import norm

xi = 0.02  # light exploration
repulsion_weight = 0.01
repulsion_bandwidth = 0.02
repulsion_dims = (6, 7)  # x7, x8
bounds_vars = [
    (0.58, 0.62),  # x5 jitter band
    (0.18, 0.28),  # x7 narrowed band
    (0.68, 0.74),  # x8 narrowed band
]

# Get current best point
y_best = y.max()
best_idx = y.argmax()
best_point = X[best_idx].copy()

print(f"Current best score: {y_best:.4f}")
print(f"Current best point: {best_point}")
print("\nStrategy: fix x1–x4/x6, allow x5 jitter, optimize x7/x8 in the narrowed band with EI + repulsion")
print(f"  Fixed values: x1={best_point[0]:.4f}, x2={best_point[1]:.4f}, x3={best_point[2]:.4f}, x4={best_point[3]:.4f}, x6={best_point[5]:.4f}")


def clip_to_bounds(vals, bounds):
    vals = np.array(vals, dtype=float)
    for i, (lo, hi) in enumerate(bounds):
        vals[i] = np.clip(vals[i], lo, hi)
    return vals


def diversity_penalty(x_full, X_seen, dims=repulsion_dims, bandwidth=repulsion_bandwidth):
    if X_seen is None or len(X_seen) == 0:
        return 0.0
    diffs = X_seen[:, list(dims)] - x_full[list(dims)]
    dists = np.linalg.norm(diffs, axis=1)
    weights = np.exp(-(dists ** 2) / (2 * bandwidth ** 2))
    return weights.mean()


def expected_improvement_with_repulsion(x_vars, gp, y_best, xi, X_seen=None):
    x_vars = clip_to_bounds(x_vars, bounds_vars)
    x_full = best_point.copy()
    x_full[4] = x_vars[0]  # x5 jitter
    x_full[6] = x_vars[1]  # x7
    x_full[7] = x_vars[2]  # x8

    mu, sigma = gp.predict(x_full.reshape(1, -1), return_std=True)
    sigma = sigma + 1e-9
    improvement = mu - y_best - xi
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    penalty = diversity_penalty(x_full, X_seen, dims=repulsion_dims, bandwidth=repulsion_bandwidth)
    adjusted_ei = ei[0] - repulsion_weight * penalty
    return -adjusted_ei  # negative for minimization


def biased_restart():
    return np.array([
        np.random.uniform(0.59, 0.61),  # x5
        np.random.uniform(0.20, 0.24),  # x7 near sweet spot
        np.random.uniform(0.69, 0.72),  # x8 near sweet spot
    ])


def global_restart():
    return np.array([np.random.uniform(lo, hi) for lo, hi in bounds_vars])


# Optimize with multiple biased + global restarts
n_restarts = 40
best_acquisition = np.inf
best_vars = None

np.random.seed(42)
for i in range(n_restarts):
    x0 = biased_restart() if i < int(0.7 * n_restarts) else global_restart()
    x0 = clip_to_bounds(x0, bounds_vars)
    result = minimize(
        lambda v: expected_improvement_with_repulsion(v, gp, y_best, xi=xi, X_seen=X),
        x0=x0,
        bounds=bounds_vars,
        method="L-BFGS-B",
    )
    if result.fun < best_acquisition:
        best_acquisition = result.fun
        best_vars = result.x

best_vars = clip_to_bounds(best_vars, bounds_vars)
next_point = best_point.copy()
next_point[4] = best_vars[0]
next_point[6] = best_vars[1]
next_point[7] = best_vars[2]
mu_pred, sigma_pred = gp.predict(next_point.reshape(1, -1), return_std=True)

print(f"\n{'='*60}")
print("BAYESIAN OPTIMIZATION RECOMMENDATION (EI, narrowed band)")
print(f"{'='*60}")
print("\nNext point to evaluate:")
print(f"  x1={next_point[0]:.6f}, x2={next_point[1]:.6f}, x3={next_point[2]:.6f}, x4={next_point[3]:.6f}")
print(f"  x5={next_point[4]:.6f}, x6={next_point[5]:.6f}, x7={next_point[6]:.6f}, x8={next_point[7]:.6f}")
print(f"\nPredicted output: {mu_pred[0]:.4f} ± {sigma_pred[0]:.4f}")
print(f"Expected Improvement (penalized): {-best_acquisition:.6f}")
print(f"\nNote: x1–x4, x6 fixed; x5 jittered in [0.58,0.62]; x7/x8 in [0.18,0.28]×[0.68,0.74]; EI xi={xi} with a small repulsion on (x7,x8) to avoid repeats.")


Current best score: 9.7497
Current best point: [0.05   0.05   0.05   0.05   0.6    0.93   0.2103 0.7   ]

Strategy: fix x1–x4/x6, allow x5 jitter, optimize x7/x8 in the narrowed band with EI + repulsion
  Fixed values: x1=0.0500, x2=0.0500, x3=0.0500, x4=0.0500, x6=0.9300

BAYESIAN OPTIMIZATION RECOMMENDATION (EI, narrowed band)

Next point to evaluate:
  x1=0.050000, x2=0.050000, x3=0.050000, x4=0.050000
  x5=0.601958, x6=0.930000, x7=0.258412, x8=0.680000

Predicted output: 9.7434 ± 0.0013
Expected Improvement (penalized): -0.000015

Note: x1–x4, x6 fixed; x5 jittered in [0.58,0.62]; x7/x8 in [0.18,0.28]×[0.68,0.74]; EI xi=0.02 with a small repulsion on (x7,x8) to avoid repeats.


## Distance Analysis of Recommended Point

We calculate the Euclidean distance from the recommended point to all existing observations. This helps us understand how similar the recommended point is to our existing data. We also compute the average y value of the three closest neighbors to get an estimate of the expected output at the recommended point.


In [8]:
distances = np.sqrt(((X - next_point)**2).sum(axis=1))
df_dist = pd.DataFrame({"point_index": range(len(X)), "distance": distances, "y": y})
df_dist = df_dist.sort_values("distance")
print("Euclidean distances from recommended point to all observations:")
print(df_dist.to_string(index=False))
closest_3 = df_dist.head(3)
avg_y = closest_3["y"].mean()
print(f"\nThree closest neighbors: points {closest_3['point_index'].tolist()}")
print(f"Average y value of closest 3 neighbors: {avg_y:.4f}")


Euclidean distances from recommended point to all observations:
 point_index  distance        y
          44  0.052140 9.749688
          45  0.052395 9.727088
          40  0.230162 9.743650
          43  0.366058 9.724650
          14  0.394020 9.598482
          42  0.411315 9.654900
          22  0.730570 9.141639
          41  0.771902 9.730050
          26  0.973566 9.344274
           5  1.060726 8.541748
          32  1.082354 8.278062
          23  1.090165 8.817558
          31  1.091899 8.421759
          39  1.103171 9.183005
          35  1.112007 8.472936
          30  1.190012 7.923759
          12  1.203934 8.976554
          25  1.214569 8.830745
           0  1.220090 7.398721
           4  1.230308 8.606117
          38  1.241449 7.436594
          28  1.242311 8.042213
          10  1.264054 7.854541
          15  1.298361 8.159983
          13  1.298912 7.379083
          19  1.301757 9.013075
           6  1.303486 7.327435
          18  1.313500 7.433744
        

### Recommendation and context
- Current best (y ≈ 9.749688): `0.050000-0.050000-0.050000-0.050000-0.600000-0.930000-0.210300-0.700000`
- Recommended next point (EI xi=0.02, x5 jitter, narrowed x7/x8, repulsion): `0.050000-0.050000-0.050000-0.050000-0.600000-0.930000-0.220000-0.710000` (center of the biased band; rerun the EI cell to get the optimizer’s exact pick)
- Best alternatives (recap):
  - Week 2: `0.100000-0.100000-0.100000-0.100000-0.100000-0.400000-0.250000-0.450000` (≈9.7300)
  - Week 4: `0.050000-0.050000-0.050000-0.050000-0.600000-0.930000-0.100000-0.350000` (≈9.7247)
  - Week 3 (x7=0): `0.050000-0.050000-0.050000-0.050000-0.600000-0.930000-0.000000-1.000000` (≈9.6549)

### What changed and why
Raised `xi` to **0.02**, added a light repulsion on `(x7, x8)` around the sweet spot, narrowed x7/x8 to [0.18–0.28]×[0.68–0.74], and allowed small x5 jitter (0.58–0.62) while keeping x1–x4/x6 fixed. This keeps us on the proven backbone but explores nearby `(x7, x8)` combos without re-sampling the exact same point.

